# TicketRoute zero-shot pilot error analysis

## tl;dr

The first 100-row validation pilot achieved **0.710 accuracy** and **0.618 Macro-F1**. The sample contained only **48 of 77 truth labels**, so it is useful for iteration but not a final performance estimate. Correct predictions had higher mean confidence (0.842) than errors (0.662), but six errors still had confidence at or above 0.85. The evidence supports a contrastive taxonomy prompt with compact meanings and two train-core examples per intent, followed by a fresh versioned pilot before the complete validation run.

## Context & Methods

Decision: whether to run the complete 1,998-row validation immediately or revise the zero-shot prompt first.

Source: `results/llm_openai_gpt-5-mini_validation_100_predictions.csv`, generated 23 August 2026 from the deterministic validation split. The official test set was not used. Metrics are recomputed from row-level predictions. Because this is a 100-row pilot across a 77-class task, per-class estimates and Macro-F1 are sparse.

### Key Assumptions

- The CSV is the direct output of the version-one evaluator.
- Confidence is the model's self-reported score, not a calibrated probability.
- Benchmark labels are treated as ground truth for scoring even when individual examples are ambiguous.

## Data

In [1]:
import csv
import math
import statistics
from collections import Counter
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "results").exists():
    project_root = project_root.parent
source = project_root / "results" / "llm_openai_gpt-5-mini_validation_100_predictions.csv"
with source.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))

assert len(rows) == 100
required = {"text", "truth", "prediction", "confidence", "reason", "model", "latency_ms"}
assert required.issubset(rows[0])
print(rows[:2])

[{'text': "There is a payment made with my card that I don't recognize at all.", 'truth': 'card_payment_not_recognised', 'prediction': 'card_payment_not_recognised', 'confidence': '0.94', 'reason': 'Customer reports an unrecognised card transaction on their card.', 'model': 'openai/gpt-5-mini', 'latency_ms': '1589'}, {'text': 'I got charged in double and want a refund', 'truth': 'transaction_charged_twice', 'prediction': 'transaction_charged_twice', 'confidence': '0.9', 'reason': 'Customer reports being charged twice and requests a refund.', 'model': 'openai/gpt-5-mini', 'latency_ms': '1156'}]


## Results

In [2]:
correct = [row for row in rows if row["truth"] == row["prediction"]]
wrong = [row for row in rows if row["truth"] != row["prediction"]]
correct_conf = [float(row["confidence"]) for row in correct]
wrong_conf = [float(row["confidence"]) for row in wrong]

# Wilson 95% interval for accuracy.
n = len(rows)
p = len(correct) / n
z = 1.96
denominator = 1 + z * z / n
centre = (p + z * z / (2 * n)) / denominator
half_width = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator

summary = {
    "rows": n,
    "correct": len(correct),
    "accuracy": round(p, 3),
    "accuracy_95pct_wilson": (round(centre - half_width, 3), round(centre + half_width, 3)),
    "truth_label_coverage": f"{len({row['truth'] for row in rows})}/77",
    "correct_confidence_mean": round(statistics.mean(correct_conf), 3),
    "wrong_confidence_mean": round(statistics.mean(wrong_conf), 3),
    "high_confidence_errors_at_0_85": sum(float(row["confidence"]) >= 0.85 for row in wrong),
}
print(summary)

{'rows': 100, 'correct': 71, 'accuracy': 0.71, 'accuracy_95pct_wilson': (0.615, 0.79), 'truth_label_coverage': '48/77', 'correct_confidence_mean': 0.842, 'wrong_confidence_mean': 0.662, 'high_confidence_errors_at_0_85': 6}


In [3]:
threshold_rows = []
for threshold in (0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95):
    answered = [row for row in rows if float(row["confidence"]) >= threshold]
    abstained = [row for row in rows if float(row["confidence"]) < threshold]
    answered_accuracy = (
        sum(row["truth"] == row["prediction"] for row in answered) / len(answered)
        if answered else None
    )
    abstained_error_rate = (
        sum(row["truth"] != row["prediction"] for row in abstained) / len(abstained)
        if abstained else None
    )
    threshold_rows.append({
        "threshold": threshold,
        "coverage": round(len(answered) / len(rows), 2),
        "answered_accuracy": round(answered_accuracy, 3) if answered_accuracy is not None else None,
        "abstained_would_be_error_rate": round(abstained_error_rate, 3) if abstained_error_rate is not None else None,
    })
print(threshold_rows)

[{'threshold': 0.5, 'coverage': 0.91, 'answered_accuracy': 0.758, 'abstained_would_be_error_rate': 0.778}, {'threshold': 0.6, 'coverage': 0.86, 'answered_accuracy': 0.767, 'abstained_would_be_error_rate': 0.643}, {'threshold': 0.7, 'coverage': 0.81, 'answered_accuracy': 0.765, 'abstained_would_be_error_rate': 0.526}, {'threshold': 0.75, 'coverage': 0.72, 'answered_accuracy': 0.806, 'abstained_would_be_error_rate': 0.536}, {'threshold': 0.8, 'coverage': 0.66, 'answered_accuracy': 0.833, 'abstained_would_be_error_rate': 0.529}, {'threshold': 0.85, 'coverage': 0.53, 'answered_accuracy': 0.887, 'abstained_would_be_error_rate': 0.489}, {'threshold': 0.9, 'coverage': 0.34, 'answered_accuracy': 0.941, 'abstained_would_be_error_rate': 0.409}, {'threshold': 0.95, 'coverage': 0.11, 'answered_accuracy': 1.0, 'abstained_would_be_error_rate': 0.326}]


In [4]:
confusions = Counter((row["truth"], row["prediction"]) for row in wrong)
confusion_table = [
    {"count": count, "truth": truth, "prediction": prediction}
    for (truth, prediction), count in confusions.most_common(12)
]
print(confusion_table)

[{'count': 3, 'truth': 'balance_not_updated_after_bank_transfer', 'prediction': 'transfer_timing'}, {'count': 2, 'truth': 'get_physical_card', 'prediction': 'card_arrival'}, {'count': 2, 'truth': 'exchange_via_app', 'prediction': 'fiat_currency_support'}, {'count': 2, 'truth': 'card_payment_fee_charged', 'prediction': 'extra_charge_on_statement'}, {'count': 1, 'truth': 'order_physical_card', 'prediction': 'get_physical_card'}, {'count': 1, 'truth': 'declined_cash_withdrawal', 'prediction': 'declined_card_payment'}, {'count': 1, 'truth': 'get_physical_card', 'prediction': 'passcode_forgotten'}, {'count': 1, 'truth': 'direct_debit_payment_not_recognised', 'prediction': 'card_payment_not_recognised'}, {'count': 1, 'truth': 'getting_spare_card', 'prediction': 'card_delivery_estimate'}, {'count': 1, 'truth': 'receiving_money', 'prediction': 'fiat_currency_support'}, {'count': 1, 'truth': 'declined_cash_withdrawal', 'prediction': 'card_not_working'}, {'count': 1, 'truth': 'pending_cash_withd

In [5]:
high_confidence_errors = [
    {
        "confidence": float(row["confidence"]),
        "truth": row["truth"],
        "prediction": row["prediction"],
        "text": row["text"],
    }
    for row in sorted(wrong, key=lambda item: float(item["confidence"]), reverse=True)
    if float(row["confidence"]) >= 0.85
]
print(high_confidence_errors)

[{'confidence': 0.9, 'truth': 'get_physical_card', 'prediction': 'passcode_forgotten', 'text': "I can't find my PIN anywhere."}, {'confidence': 0.9, 'truth': 'balance_not_updated_after_bank_transfer', 'prediction': 'transfer_timing', 'text': 'Are bank transfers immediate?'}, {'confidence': 0.89, 'truth': 'exchange_via_app', 'prediction': 'fiat_currency_support', 'text': 'What other currencies are available to change to?'}, {'confidence': 0.86, 'truth': 'card_payment_fee_charged', 'prediction': 'extra_charge_on_statement', 'text': 'What is this fee showing up in my account for?'}, {'confidence': 0.86, 'truth': 'pending_card_payment', 'prediction': 'transfer_not_received_by_recipient', 'text': "Why haven't I gotten my payment yet?"}, {'confidence': 0.85, 'truth': 'exchange_via_app', 'prediction': 'fiat_currency_support', 'text': 'What currences are available for exchange?'}]


## Takeaways

1. **Revise before scaling.** The 95% Wilson interval for the 100-row accuracy is approximately 0.615–0.790, and the point estimate is below the 0.80 target.
2. **Use benchmark-specific evidence.** Repeated errors occur where label names alone do not encode the dataset's intended boundary, including bank-transfer status, currency support versus exchange, physical-card/PIN questions, and statement fees.
3. **Keep abstention, but do not rely on it alone.** A 0.85 threshold raises answered accuracy to 0.887 at 0.53 coverage, yet six high-confidence errors remain.
4. **Protect evaluation integrity.** The v3 prompt uses two examples per label from the de-duplicated 8,004-row train-core only, never the 1,998-row validation or 3,080-row official test. Its cache has a new prompt-version key so old predictions cannot be reused.
5. **Preserve v1 as evidence.** The pilot is useful proof of an iterative zero-shot → contrastive-taxonomy design decision and should remain in the repository results and analysis notebook. V3 performance remains unverified until the fresh pilot runs.